# Classification: Trial-Level Bias Score (TL-BS)

Based on Zvielli, Bernstein & Koster (2014), attentional bias is treated as a dynamic signal rather than a stable trait. Instead of averaging fixation bias across all trials, we extract parameters that describe the shape and dynamics of the bias signal across the session: mean positive/negative bias, peak values, trial-to-trial variability, and linear slope. These TL-BS parameters are then compared against traditional session-level means as classification features.

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from src.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Load trial-level data

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

df_trials = numbered.select("session_id", "trial_num", "fixation_bias").toPandas()

MEAN_METRICS = [
    "fixation_count", "mean_fixation_duration_ms", "fixation_rate_per_sec",
    "fixation_bias", "scanpath_length", "saccade_count",
    "blink_count", "blink_rate_per_min",
    "dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral",
    "fixation_proportion_negative", "fixation_proportion_positive", "fixation_proportion_neutral",
    "revisit_count_negative", "revisit_count_positive",
    "ttff_ms_negative", "ttff_ms_positive",
    "first_fixation_duration_ms",
]

traditional_aggs = [F.mean(m).alias(f"avg_{m}") for m in MEAN_METRICS]
for valence in ["negative", "positive", "neutral"]:
    traditional_aggs.append(
        F.avg(F.when(F.col("first_fixation_valence") == valence, 1).otherwise(0))
         .alias(f"first_fix_prob_{valence}"))

df_traditional = df_stimulus.groupBy("session_id").agg(*traditional_aggs).toPandas()

print(f"Trial-level data: {len(df_trials)} rows, {df_trials['session_id'].nunique()} sessions")

## 2. Compute TL-BS parameters

In [0]:
def compute_tlbs_params(group):
    """
    Compute Zvielli-style TL-BS parameters from fixation_bias trial series.
    Positive bias = more attention to negative stimuli.
    Negative bias = more attention to positive stimuli.
    """
    values = group["fixation_bias"].dropna().values
    trial_nums = group.loc[group["fixation_bias"].notna(), "trial_num"].values

    if len(values) < 3:
        return {"tlbs_mean_pos": np.nan, "tlbs_mean_neg": np.nan,
                "tlbs_peak_pos": np.nan, "tlbs_peak_neg": np.nan,
                "tlbs_variability": np.nan, "tlbs_slope": np.nan}

    pos_vals = values[values > 0]
    neg_vals = values[values < 0]

    slope, _, _, _, _ = scipy_stats.linregress(trial_nums, values)

    return {
        "tlbs_mean_pos": float(np.mean(pos_vals)) if len(pos_vals) > 0 else 0.0,
        "tlbs_mean_neg": float(np.mean(neg_vals)) if len(neg_vals) > 0 else 0.0,
        "tlbs_peak_pos": float(np.max(values)),
        "tlbs_peak_neg": float(np.min(values)),
        "tlbs_variability": float(np.mean(np.abs(np.diff(values)))),
        "tlbs_slope": float(slope),
    }

tlbs_rows = []
for session_id, group in df_trials.groupby("session_id"):
    group = group.sort_values("trial_num")
    row = {"session_id": session_id}
    row.update(compute_tlbs_params(group))
    tlbs_rows.append(row)

df_tlbs = pd.DataFrame(tlbs_rows)
print(f"TL-BS features computed for {len(df_tlbs)} sessions")

## 3. Visualize TL-BS dynamics by severity

In [0]:
df_forms_viz = df_forms.select("session_id", "phq9_score", "phq9_severity").toPandas()
df_viz = df_trials.merge(df_forms_viz, on="session_id")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)

for ax, sev in zip(axes, ["minimal", "moderate", "severe"]):
    subset = df_viz[df_viz["phq9_severity"] == sev]
    sample_sessions = subset["session_id"].unique()[:5]
    for sid in sample_sessions:
        vals = subset[subset["session_id"] == sid].sort_values("trial_num")["fixation_bias"].values
        ax.plot(range(len(vals)), vals, alpha=0.5, linewidth=1.5)
    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"PHQ-9: {sev}")
    ax.set_xlabel("Trial number")
    ax.set_ylabel("Fixation bias" if ax == axes[0] else "")

plt.suptitle("Trial-level fixation bias signal", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Build dataset and define feature sets

In [0]:
df = df_tlbs.merge(df_traditional, on="session_id")
df = df.merge(
    df_forms.select("session_id", "uid", "phq9_score", "phq9_severity").toPandas(),
    on="session_id",
)

df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)
df["phq9_severity_class"] = df["phq9_severity"].map(
    {"minimal": 0, "mild": 1, "moderate": 2, "moderately_severe": 3, "severe": 4}
)

print(f"Sessions: {len(df)}, Users: {df['uid'].nunique()}")
print(f"Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")

In [0]:
TLBS_FEATURES = [
    "tlbs_mean_pos", "tlbs_mean_neg",
    "tlbs_peak_pos", "tlbs_peak_neg",
    "tlbs_variability", "tlbs_slope",
]

TRADITIONAL_FEATURES = [f"avg_{m}" for m in MEAN_METRICS] + [
    "first_fix_prob_negative", "first_fix_prob_positive", "first_fix_prob_neutral",
]

COMBINED_FEATURES = TRADITIONAL_FEATURES + TLBS_FEATURES

FEATURE_SETS = {
    "TL-BS Only": TLBS_FEATURES,
    "Traditional Means": TRADITIONAL_FEATURES,
    "Combined": COMBINED_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 5. Prepare data

In [0]:
target_cols = ["phq9_score", "phq9_depressed", "phq9_severity_class"]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 6. TL-BS correlations with PHQ-9

In [0]:
print("TL-BS parameter correlations with PHQ-9:\n")
for feat in TLBS_FEATURES:
    r, p = scipy_stats.spearmanr(df_clean[feat], df_clean["phq9_score"])
    star = "*" if p < 0.05 else " "
    print(f"  {star} r={r:+.3f}  p={p:.2e}  {feat}")

r_trad, p_trad = scipy_stats.spearmanr(df_clean["avg_fixation_bias"], df_clean["phq9_score"])
print()
print(f"Reference: avg_fixation_bias  r={r_trad:+.3f}  p={p_trad:.2e}")

## 7. Binary classification (PHQ-9 >= 10)

### 7.1 Run classification

In [0]:
y_binary = df_clean["phq9_depressed"].values
binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_binary, groups)

### 7.2 Results

In [0]:
print(binary_df.to_string(index=False))

### 7.2 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_binary, groups, binary_df)

## 8. Multi-class classification (5 severity groups)

### 8.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_multi = df_clean["phq9_severity_class"].values.astype(int)
multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups)

### 8.2 Results

In [0]:
print(multi_df.to_string(index=False))

### 8.2 Best model


In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups, multi_df, PHQ9_LABELS)

## 9. Regression (predict PHQ-9 score)

### 9.1 Run regression

In [0]:
y_reg = df_clean["phq9_score"].values
reg_df = run_regression(df_clean, FEATURE_SETS, y_reg, groups)

### 9.2 Results

In [0]:
print(reg_df.to_string(index=False))

### 9.2 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_reg, groups, reg_df)

## 10. Feature importance

In [0]:
plot_feature_importance(df_clean, COMBINED_FEATURES, y_binary, title="Feature importance (binary, combined)")

## 11. Summary

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(binary_df, multi_df, reg_df, feature_order, title="TL-BS features")